In [ ]:
import pandas as pd
import geopandas as gpd
import glob
import os


Faça o tratamento dos dfs de vcs definindo uma funcao aqui embaixo, e depois depois adicionem a funcao no mapa_tratamento falando o nome do arquivo correspondente. Fiz o tratar_ids como exemplo. Acho que assim vai ficar mais organizado

In [ ]:
def tratar_ids(df):
    return df[['codbairro', 'bairro', 'ids']]

Tratamento de datasets do gtfs para cria a matriz de pares (linha,bairo)

In [ ]:
def obter_matriz_linha_bairo(df_routes, df_stops, df_route_stops):
    df_routes = df_routes.drop_duplicates(subset=['route_id', 'versao_modelo'])
    df_routes = df_routes[df_routes['route_type'] != 702]
    df_routes = df_routes[~df_routes['route_short_name'].str.contains('LECD|ESP01', na=False)]
    df_routes['route_short_name'] = df_routes['route_short_name'].str.lstrip('0')

    df_stops = df_stops.drop_duplicates(subset=['stop_id', 'versao_modelo'])

    df_route_stops = df_route_stops[df_route_stops['service_id'].isin(['D', 'D_REG', 'S', 'S_REG', 'U', 'U_REG'])]
    df_route_stops = df_route_stops.drop_duplicates(['stop_id', 'route_id', 'versao_modelo']).drop(['service_id'], axis=1)

    df_pipeline = pd.merge(
        df_route_stops,
        df_routes[['route_id', 'versao_modelo', 'route_short_name']],
        on=['route_id', 'versao_modelo'],
        how='inner'
    )

    df_pipeline = pd.merge(
        df_pipeline,
        df_stops[['stop_id', 'versao_modelo', 'stop_lat', 'stop_lon']],
        on=['stop_id', 'versao_modelo'],
        how='inner'
    )

    df_pipeline = df_pipeline.drop_duplicates().drop(['route_id', 'stop_id', 'versao_modelo'], axis=1).rename(columns={'route_short_name': 'linha', 'stop_lat': 'latitude', 'stop_lon': 'longitude'})

    gdf_onibus = gpd.GeoDataFrame(
        df_pipeline,
        geometry=gpd.points_from_xy(df_pipeline.longitude, df_pipeline.latitude),
        crs="EPSG:4326"
    )

    gdf_bairros = gpd.read_file('../dados/Limite_de_Bairros.geojson')

    gdf_bairros = gdf_bairros.to_crs("EPSG:4326")

    resultado = gpd.sjoin(gdf_onibus, gdf_bairros, how="inner", predicate="intersects")

    matriz = (
        resultado[['nome', 'linha']]
        .drop_duplicates()
        .rename(columns={'nome': 'bairro'})
        .sort_values(['bairro', 'linha'])
        .reset_index(drop=True)
    )

    return matriz

Tratamento dos datasets de lotação dos onibus e prestação dos serviços e classificação das Zonas

In [ ]:
def analisar_lotacao_regiao(df_lotacao, df_sumario_servico, df_linha_bairro):
    df_raw= df_lotacao

    df_raw['servico'] =df_raw['servico'].astype(str) #servico (linha) está como número mas é pra ser interpretado como string

    lot=df_raw[(df_raw['servico'] !='795') & (~df_raw['servico'].str.contains('LECD'))].copy()
    lot['data']=pd.to_datetime(lot['data'],format='%d/%m/%Y') #coloca a coluna data com valor de data no formato certo

    sum_= df_sumario_servico
    sum_['data']=pd.to_datetime(sum_['data'],format='%Y-%m-%d')

    mb = df_linha_bairro # colunas: 'bairro' e 'linha'
    mb['linha']=mb['linha'].astype(str)

    PERIODO=f"{sum_['data'].min().date()} a {sum_['data'].max().date()}"

    lot = lot[lot['class_servico'] != 'Rodoviário'] #remove do lotação onibus rodoviários

    # limpar o sumário
    sum_=sum_[sum_['km_planejada']>0] #linha não era pra operar no dia
    sum_=sum_[['servico','data','perc_km_planejada']] #só precisa dessas 3 colunas

    df=lot.merge(sum_,on=['servico', 'data'],how='inner') #merge de lotacao e sumario onde servico e data iguais em ambos

    # Classificar região
    zona_oeste = [
        'Campo Grande','Santa Cruz','Bangu','Realengo','Padre Miguel',
        'Sepetiba','Guaratiba','Paciência','Santíssimo','Inhoaíba',
        'Cosmos','Pedra de Guaratiba','Ilha de Guaratiba','Jabour',
        'Jardim América','Magalhães Bastos','Senador Camará',
        'Senador Vasconcelos','Vila Kennedy','Deodoro','Campo dos Afonsos',
        'Vila Militar','Anchieta','Recreio dos Bandeirantes',
        'Barra da Tijuca','Barra de Guaratiba','Barra Olímpica',
        'Vargem Grande','Vargem Pequena','Camorim','Curicica',
        'Gardênia Azul','Itanhangá','Jacarepaguá','Pechincha',
        'Praça Seca','Taquara','Tanque','Anil','Cidade de Deus',
        'Jardim Sulacap','Vila Valqueire','Freguesia (Jacarepaguá)',
    ]
    zona_norte = [
        'Madureira','Cascadura','Irajá','Pavuna','Marechal Hermes',
        'Méier','Engenho de Dentro','Engenho da Rainha','Engenho Novo',
        'Bonsucesso','Ramos','Olaria','Penha','Cordovil','Guadalupe',
        'Rocha Miranda','Honório Gurgel','Coelho Neto','Bento Ribeiro',
        'Del Castilho','Inhaúma','Acari','Água Santa',
        'Grajaú','Vila Isabel','Andaraí','Tijuca','Alto da Boa Vista',
        'Cachambi','Campinho','Encantado','Lins de Vasconcelos','Pilares',
        'Piedade','Quintino Bocaiúva','Riachuelo','Rocha','Sampaio',
        'Todos os Santos','Tomás Coelho','Turiaçu','Osvaldo Cruz',
        'Barros Filho','Colégio','Costa Barros','Engenheiro Leal',
        'Higienópolis','Parque Colúmbia','Parada de Lucas',
        'Ricardo de Albuquerque','Vaz Lobo','Vicente de Carvalho',
        'Vila da Penha','Vista Alegre','Vigário Geral',
        'Complexo do Alemão','Jacarezinho','Manguinhos','Benfica',
        'Caju','Mangueira','Maracanã','São Francisco Xavier',
        'Jacaré','Brás de Pina','Cavalcanti','Penha Circular',
        'Vasco da Gama','Maria da Graça','Bancários','Cacuia','Cocotá',
        'Galeão','Jardim Carioca','Jardim Guanabara','Moneró',
        'Pitangueiras','Portuguesa','Praia da Bandeira','Ribeira',
        'Tauá','Zumbi','Maré','Abolição', 'Cidade Universitária', 'Freguesia (Ilha)'
    ]
    zona_sul = [
        'Ipanema','Leblon','Botafogo','Copacabana','Gávea','Lagoa',
        'Leme','Flamengo','Cosme Velho','Humaitá','São Conrado',
        'Jardim Botânico','Rocinha','Vidigal','Urca','Laranjeiras',
        'Catete','Glória','Santa Teresa','Joá'
    ]
    centro = [
        'Centro','Cidade Nova','Santo Cristo','Gamboa','Saúde',
        'Rio Comprido','Imperial de São Cristóvão','Praça da Bandeira',
        'Catumbi','Estácio','Lapa'
    ]


    def classificar_regiao(bairro):
        if bairro in zona_sul: return 'Zona Sul'
        if bairro in centro: return 'Centro'
        if bairro in zona_norte: return 'Zona Norte / Subúrbio'
        if bairro in zona_oeste: return 'Zona Oeste'
        return 'Não identificado'

    # Classificar todos os bairros da matriz_linha, adiciona uma nova coluna regiao
    mb['regiao'] = mb['bairro'].apply(classificar_regiao)

    # Para cada linha, pegar a região mais frequente entre todos os bairros dela
    regiao_por_linha = (
        mb[mb['regiao'] != 'Não identificado']
        .groupby('linha')['regiao']
        .agg(lambda x: x.value_counts().index[0])
        .reset_index()
        .rename(columns={'linha': 'servico'}) # renomear só aqui para o merge
    )

    df['servico']=df['servico'].astype(str) #colocar a linha do onibus como string
    regiao_por_linha['servico']=regiao_por_linha['servico'].astype(str)
    df=df.merge(regiao_por_linha, on='servico', how='left') #merge left, encaixa o regiao por linha no df mesmo se não tiver correspondente no região por linha
    df['regiao'] = df['regiao'].fillna('Não identificado') #preenche os nan com nao identificado


    df['mes'] = df['data'].dt.to_period('M') #cria nova coluna mês, que só pega o ano e o mês da data (2023-07-25 → 2023-07)
    #agrupa pelos os que tem o msm servico(linha), mes e regiao
    #agg(coluna=('colunax','método') cria colunas novas usando colunas do df mensal agrupado e fazendo alguma operação, unique conta valores unicos, mean média, sum soma tudo)
    df_mensal = df.groupby(['servico', 'mes', 'regiao']).agg(
        passageiros_total=('qtd_passageiros_total', 'sum'),
        viagens_total=('qtd_viagens', 'sum'),
        cumprimento=('perc_km_planejada', 'mean')).reset_index()
    df_mensal['passageiros_por_viagem']=(df_mensal['passageiros_total']/df_mensal['viagens_total'].replace(0, None)) #adiciona essa coluna passageiros por viagens usando a coluna do df viagens total que acabou de ser criada

    # limpeza simples onde tira outliers absurdos
    df_mensal = df_mensal[df_mensal['passageiros_por_viagem'] <= 250]

    # cria a análise por região aqui. df_mensal.groupby('regiao') junta todas as linhas da mesma região
    por_regiao = df_mensal.groupby('regiao').agg(
        linhas=('servico','nunique'),
        passageiros_por_viagem=('passageiros_por_viagem','mean'),
        cumprimento=('cumprimento', 'mean'),
        total_passageiros=('passageiros_total','sum')
    ).round(1).sort_values('cumprimento').reset_index() #tinha transformado regiao em indice, reset index volta o regiao como coluna

    por_regiao.insert(0, 'periodo', PERIODO) #coloca o periodo na coluna 0 (no começo)

    #mesma coisa de antes
    ranking = df_mensal.groupby(['servico','regiao']).agg(
        meses=('mes','nunique'),
        cumprimento_medio=('cumprimento','mean'),
        passageiros_por_viagem=('passageiros_por_viagem','mean'),
        total_passageiros=('passageiros_total','sum')
    ).reset_index()
    ranking=ranking[ranking['meses'] >= 10] #amostra de 10 meses pra cima
    ranking =ranking.sort_values('cumprimento_medio').head(20).reset_index(drop=True) #só mostra os 20 primeiros
    ranking.index+= 1
    ranking.insert(0, 'periodo', PERIODO)
    ranking =ranking.round(1)

    return por_regiao, df_mensal, ranking




In [ ]:
def analisar_lotacao_regiao(df_lotacao, df_sumario_servico, df_linha_bairro):
    df_raw= df_lotacao

    df_raw['servico'] =df_raw['servico'].astype(str) #servico (linha) está como número mas é pra ser interpretado como string

    lot=df_raw[(df_raw['servico'] !='795') & (~df_raw['servico'].str.contains('LECD'))].copy()
    lot['data']=pd.to_datetime(lot['data'],format='%d/%m/%Y') #coloca a coluna data com valor de data no formato certo

    sum_= df_sumario_servico
    sum_['data']=pd.to_datetime(sum_['data'],format='%Y-%m-%d')
    mb = df_linha_bairro # colunas: 'bairro' e 'linha'
    mb['linha']=mb['linha'].astype(str)

    PERIODO=f"{sum_['data'].min().date()} a {sum_['data'].max().date()}"

    lot = lot[lot['class_servico'] != 'Rodoviário'] #remove do lotação onibus rodoviários

    # limpar o sumário
    sum_=sum_[sum_['km_planejada']>0] #linha não era pra operar no dia
    sum_=sum_[['servico','data','perc_km_planejada']] #só precisa dessas 3 colunas
    sum_= sum_[~((sum_['perc_km_planejada']<10) & (sum_['perc_km_planejada']>200))]
    df=lot.merge(sum_,on=['servico', 'data'],how='inner') #merge de lotacao e sumario onde servico e data iguais em ambos

    # Classificar região
    zona_oeste = [
        'Campo Grande','Santa Cruz','Bangu','Realengo','Padre Miguel',
        'Sepetiba','Guaratiba','Paciência','Santíssimo','Inhoaíba',
        'Cosmos','Pedra de Guaratiba','Ilha de Guaratiba','Jabour',
        'Jardim América','Magalhães Bastos','Senador Camará',
        'Senador Vasconcelos','Vila Kennedy','Deodoro','Campo dos Afonsos',
        'Vila Militar','Anchieta','Recreio dos Bandeirantes',
        'Barra da Tijuca','Barra de Guaratiba','Barra Olímpica',
        'Vargem Grande','Vargem Pequena','Camorim','Curicica',
        'Gardênia Azul','Itanhangá','Jacarepaguá','Pechincha',
        'Praça Seca','Taquara','Tanque','Anil','Cidade de Deus',
        'Jardim Sulacap','Vila Valqueire','Freguesia (Jacarepaguá)',
    ]
    zona_norte = [
        'Madureira','Cascadura','Irajá','Pavuna','Marechal Hermes',
        'Méier','Engenho de Dentro','Engenho da Rainha','Engenho Novo',
        'Bonsucesso','Ramos','Olaria','Penha','Cordovil','Guadalupe',
        'Rocha Miranda','Honório Gurgel','Coelho Neto','Bento Ribeiro',
        'Del Castilho','Inhaúma','Acari','Água Santa',
        'Grajaú','Vila Isabel','Andaraí','Tijuca','Alto da Boa Vista',
        'Cachambi','Campinho','Encantado','Lins de Vasconcelos','Pilares',
        'Piedade','Quintino Bocaiúva','Riachuelo','Rocha','Sampaio',
        'Todos os Santos','Tomás Coelho','Turiaçu','Osvaldo Cruz',
        'Barros Filho','Colégio','Costa Barros','Engenheiro Leal',
        'Higienópolis','Parque Colúmbia','Parada de Lucas',
        'Ricardo de Albuquerque','Vaz Lobo','Vicente de Carvalho',
        'Vila da Penha','Vista Alegre','Vigário Geral',
        'Complexo do Alemão','Jacarezinho','Manguinhos','Benfica',
        'Caju','Mangueira','Maracanã','São Francisco Xavier',
        'Jacaré','Brás de Pina','Cavalcanti','Penha Circular',
        'Vasco da Gama','Maria da Graça','Bancários','Cacuia','Cocotá',
        'Galeão','Jardim Carioca','Jardim Guanabara','Moneró',
        'Pitangueiras','Portuguesa','Praia da Bandeira','Ribeira',
        'Tauá','Zumbi','Maré','Abolição', 'Cidade Universitária', 'Freguesia (Ilha)'
    ]
    zona_sul = [
        'Ipanema','Leblon','Botafogo','Copacabana','Gávea','Lagoa',
        'Leme','Flamengo','Cosme Velho','Humaitá','São Conrado',
        'Jardim Botânico','Rocinha','Vidigal','Urca','Laranjeiras',
        'Catete','Glória','Santa Teresa','Joá'
    ]
    centro = [
        'Centro','Cidade Nova','Santo Cristo','Gamboa','Saúde',
        'Rio Comprido','Imperial de São Cristóvão','Praça da Bandeira',
        'Catumbi','Estácio','Lapa'
    ]


    def classificar_regiao(bairro):
        if bairro in zona_sul: return 'Zona Sul'
        if bairro in centro: return 'Centro'
        if bairro in zona_norte: return 'Zona Norte / Subúrbio'
        if bairro in zona_oeste: return 'Zona Oeste'
        return 'Não identificado'

    # Classificar todos os bairros da matriz_linha, adiciona uma nova coluna regiao
    mb['regiao'] = mb['bairro'].apply(classificar_regiao)

    # Para cada linha, pegar a região mais frequente entre todos os bairros dela
    regiao_por_linha = (
        mb[mb['regiao'] != 'Não identificado']
        .groupby('linha')['regiao']
        .agg(lambda x: x.value_counts().index[0])
        .reset_index()
        .rename(columns={'linha': 'servico'}) # renomear só aqui para o merge
    )

    df['servico']=df['servico'].astype(str) #colocar a linha do onibus como string
    regiao_por_linha['servico']=regiao_por_linha['servico'].astype(str)
    df=df.merge(regiao_por_linha, on='servico', how='left') #merge left, encaixa o regiao por linha no df mesmo se não tiver correspondente no região por linha
    df['regiao'] = df['regiao'].fillna('Não identificado') #preenche os nan com nao identificado


    df['mes'] = df['data'].dt.to_period('M') #cria nova coluna mês, que só pega o ano e o mês da data (2023-07-25 → 2023-07)
    #agrupa pelos os que tem o msm servico(linha), mes e regiao
    #agg(coluna=('colunax','método') cria colunas novas usando colunas do df mensal agrupado e fazendo alguma operação, unique conta valores unicos, mean média, sum soma tudo)
    df_mensal = df.groupby(['servico', 'mes', 'regiao']).agg(
        passageiros_total=('qtd_passageiros_total', 'sum'),
        viagens_total=('qtd_viagens', 'sum'),
        cumprimento=('perc_km_planejada', 'mean')).reset_index()
    df_mensal['passageiros_por_viagem']=(df_mensal['passageiros_total']/df_mensal['viagens_total'].replace(0, None)) #adiciona essa coluna passageiros por viagens usando a coluna do df viagens total que acabou de ser criada

    # limpeza simples onde tira outliers absurdos
    df_mensal = df_mensal[df_mensal['passageiros_por_viagem'] <= 250]
    df_mensal = df_mensal[~((df_mensal['cumprimento'] < 10) & (df_mensal['viagens_total'] > 0))]
    df_mensal = df_mensal[df_mensal['cumprimento'] <= 300]

    # cria a análise por região aqui. df_mensal.groupby('regiao') junta todas as linhas da mesma região
    por_regiao = df_mensal.groupby('regiao').agg(
        linhas=('servico','nunique'),
        passageiros_por_viagem=('passageiros_por_viagem','mean'),
        cumprimento=('cumprimento', 'mean'),
        total_passageiros=('passageiros_total','sum')
    ).round(1).sort_values('cumprimento').reset_index() #tinha transformado regiao em indice, reset index volta o regiao como coluna

    por_regiao.insert(0, 'periodo', PERIODO) #coloca o periodo na coluna 0 (no começo)

    #mesma coisa de antes
    ranking = df_mensal.groupby(['servico','regiao']).agg(
        meses=('mes','nunique'),
        cumprimento_medio=('cumprimento','mean'),
        passageiros_por_viagem=('passageiros_por_viagem','mean'),
        total_passageiros=('passageiros_total','sum')
    ).reset_index()
    ranking=ranking[ranking['meses'] >= 10] #amostra de 10 meses pra cima
    ranking =ranking.sort_values('cumprimento_medio').head(20).reset_index(drop=True) #só mostra os 20 primeiros
    ranking.index+= 1
    ranking.insert(0, 'periodo', PERIODO)
    ranking =ranking.round(1)

    return por_regiao, df_mensal, ranking




Armazenamento dos dfs tratados.

In [ ]:
mapa_tratamento = {
    'ids_por_bairro_rj.csv': tratar_ids,
    # coloquem os nomes de cada arquivo e as funcoes correspondentes aqui
}

datasets = {}

# daqui em diante ele vai tratar tudo e colocar no dicionário datasets criado ali em cima
arquivos = glob.glob('../dados/*.csv')

for caminho in arquivos:
    nome_arquivo = os.path.basename(caminho)

    nome_df = os.path.splitext(nome_arquivo)[0].lower()

    df = pd.read_csv(caminho)

    if nome_arquivo in mapa_tratamento:
        df = mapa_tratamento[nome_arquivo](df)

    datasets[nome_df] = df

#colocando no dicionário dfs que precisam de mais de um dataset

df_routes = pd.read_csv('../dados/gtfs/routes.csv')
df_stops = pd.read_csv('../dados/gtfs/stops.csv')
df_route_stops = pd.read_csv('../dados/gtfs/route_stops.csv')
datasets['matriz_linha_bairro'] = obter_matriz_linha_bairo(df_routes, df_stops, df_route_stops)

df_lotacao = pd.read_csv('../dados/lotacao_onibus.csv')
df_sumario_servico = pd.read_csv('../dados/dashboard_subsidio_sppo/sumario_servico_dia_tipo.csv')
datasets['lotacao_por_regiao'], datasets['lotacao_completo'], datasets['ranking'] = analisar_lotacao_regiao(df_lotacao, df_sumario_servico, datasets['matriz_linha_bairro'])
%store datasets

Stored 'datasets' (dict)


In [ ]:
#para acessar o dataframe em outros notebooks, você precisar executar %store -r datasets em alguma célula do notebook desejado (obs, tem que dar run all primeiro).
# pra acessar um dataframe específico, só fazer datasets['nome do arquivo sem a extensão']. vou deixar aq um exemplo
datasets['ids_por_bairro_rj'].head(5)

,codbairro,bairro,ids
0,13,Paquetá,0.571297
1,98,Freguesia (Ilha),0.594303
2,97,Bancários,0.589105
3,104,Galeão,0.562762
4,101,Tauá,0.576847


In [ ]:
datasets['lotacao_completo'].info()

<class 'pandas.DataFrame'>
Index: 4759 entries, 0 to 8267
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype    
---  ------                  --------------  -----    
 0   servico                 4759 non-null   str      
 1   mes                     4759 non-null   period[M]
 2   regiao                  4759 non-null   str      
 3   passageiros_total       4759 non-null   int64    
 4   viagens_total           4759 non-null   int64    
 5   cumprimento             4759 non-null   float64  
 6   passageiros_por_viagem  4759 non-null   float64  
dtypes: float64(2), int64(2), period[M](1), str(2)
memory usage: 297.4 KB
